In [6]:
pip install tensorflow setuptools

Note: you may need to restart the kernel to use updated packages.


In [1]:
import tensorflow as tf
import numpy as np
from datetime import datetime, time

class CompressedCircadianModel:
    def __init__(self):
        self.model = self._build_model()
        self._generate_training_data()
        self.train_model()
    
    def _build_model(self):
        """Build tiny model for circadian lighting"""
        model = tf.keras.Sequential([
            tf.keras.layers.Input(shape=(2,)),  # [seconds_in_cycle, milliseconds]
            tf.keras.layers.Dense(8, activation='relu'),
            tf.keras.layers.Dense(4, activation='relu'),
            tf.keras.layers.Dense(1, activation='sigmoid')  # White LED intensity
        ])
        
        model.compile(
            optimizer='adam',
            loss='mse',
            metrics=['mae']
        )
        
        return model
    
    def _generate_training_data(self):
        """
        Generate training data for 24-minute cycle
        Each minute represents one hour of the day
        """
        # Generate time points (24 minutes * 60 seconds * 10 points per second = 14400 points)
        total_seconds = 24 * 60  # 24 minutes in seconds
        time_points = np.linspace(0, total_seconds, 14400)
        
        # Convert to hours (0-23) for intensity calculation
        hours = (time_points / 60) % 24
        
        # Normalize inputs
        self.X_train = np.column_stack([
            time_points / total_seconds,  # Normalize to 0-1
            (time_points % 60) / 60  # Normalize seconds within each minute
        ])
        
        # Generate target intensities
        intensities = []
        for hour in hours:
            # Early morning (5:00-9:00): Gradual increase
            if 5 <= hour < 9:
                progress = (hour - 5) / 4
                intensity = 0.3 + (0.7 * progress)
            
            # Mid-morning to afternoon (9:00-15:00): High intensity
            elif 9 <= hour < 15:
                intensity = 1.0
            
            # Afternoon to evening (15:00-21:00): Gradual decrease
            elif 15 <= hour < 21:
                progress = (hour - 15) / 6
                intensity = 1.0 - (0.8 * progress)
            
            # Night (21:00-5:00): Low intensity
            else:
                intensity = 0.2
            
            intensities.append(intensity)
        
        self.y_train = np.array(intensities)
    
    def train_model(self, epochs=100):
        """Train the model"""
        self.model.fit(
            self.X_train, 
            self.y_train,
            epochs=epochs,
            batch_size=32,
            verbose=1
        )
    
    def predict_intensity(self, seconds, milliseconds):
        """Predict white LED intensity for given time in compressed cycle"""
        total_seconds = 24 * 60  # 24 minutes in seconds
        x = np.array([[seconds / total_seconds, 
                      (seconds % 60) / 60]])
        return self.model.predict(x, verbose=0)[0][0]
    
    def quantize_model(self):
        """Quantize model for TinyML deployment"""
        converter = tf.lite.TFLiteConverter.from_keras_model(self.model)
        converter.optimizations = [tf.lite.Optimize.DEFAULT]
        converter.target_spec.supported_types = [tf.float16]
        tflite_model = converter.convert()
        return tflite_model

def generate_arduino_code(model_data, output_file="circadian_model.h"):
    """Generate Arduino header file with quantized model data"""
    code = """// Compressed Circadian Lighting TinyML Model
#ifndef CIRCADIAN_MODEL_H
#define CIRCADIAN_MODEL_H

// Time compression settings
#define DAY_LENGTH_MILLIS (24L * 60L * 1000L)  // 24 minutes
#define MINUTE_LENGTH_MILLIS (60L * 1000L)     // 1 minute
#define UPDATE_INTERVAL 100                     // Update every 100ms

const unsigned char model_data[] = {
"""
    
    # Add model data as hex values
    hex_data = [f"0x{b:02x}" for b in model_data]
    chunks = [hex_data[i:i+12] for i in range(0, len(hex_data), 12)]
    for chunk in chunks:
        code += "  " + ", ".join(chunk) + ",\n"
    
    code += """};

#endif
"""
    
    with open(output_file, 'w') as f:
        f.write(code)

# Example usage:
if __name__ == "__main__":
    model = CompressedCircadianModel()

    # Test predictions
    test_times = [
        (5 * 60, 0),    # 5:00
        (12 * 60, 0),   # 12:00
        (18 * 60, 0),   # 18:00
        (22 * 60, 0)    # 22:00
    ]

    print("\nPredicted White LED Intensities:")
    for seconds, ms in test_times:
        intensity = model.predict_intensity(seconds, ms)
        minutes = seconds // 60
        print(f"{minutes:02d}:00 - {intensity:.2f}")

    # Quantize and generate Arduino code
    print("\nQuantizing model and generating Arduino code...")
    tflite_model = model.quantize_model()
    generate_arduino_code(tflite_model)
    print("Done! Generated circadian_model.h")

Epoch 1/100
450/450 ━━━━━━━━━━━━━━━━━━━━ 1s 453us/step - loss: 0.1093 - mae: 0.3045
Epoch 2/100
450/450 ━━━━━━━━━━━━━━━━━━━━ 0s 413us/step - loss: 0.0863 - mae: 0.2722
Epoch 3/100
450/450 ━━━━━━━━━━━━━━━━━━━━ 0s 402us/step - loss: 0.0509 - mae: 0.2070
Epoch 4/100
450/450 ━━━━━━━━━━━━━━━━━━━━ 0s 396us/step - loss: 0.0244 - mae: 0.1358
Epoch 5/100
450/450 ━━━━━━━━━━━━━━━━━━━━ 0s 392us/step - loss: 0.0124 - mae: 0.0956
Epoch 6/100
450/450 ━━━━━━━━━━━━━━━━━━━━ 0s 397us/step - loss: 0.0066 - mae: 0.0702
Epoch 7/100
450/450 ━━━━━━━━━━━━━━━━━━━━ 0s 402us/step - loss: 0.0039 - mae: 0.0543
Epoch 8/100
450/450 ━━━━━━━━━━━━━━━━━━━━ 0s 400us/step - loss: 0.0027 - mae: 0.0441
Epoch 9/100
450/450 ━━━━━━━━━━━━━━━━━━━━ 0s 396us/step - loss: 0.0022 - mae: 0.0378
Epoch 10/100
450/450 ━━━━━━━━━━━━━━━━━━━━ 0s 392us/step - loss: 0.0019 - mae: 0.0335
Epoch 11/100
450/450 ━━━━━━━━━━━━━━━━━━━━ 0s 395us/step - loss: 0.0018 - mae: 0.0317
Epoch 12/100
450/450 ━━━━━━━━━━━━━━━━━━━━ 0s 401us/step - loss: 0.0017 - m

INFO:tensorflow:Assets written to: /var/folders/vs/xs37hl812y7_g9k12gvs5tww0000gn/T/tmp4q0m5rwp/assets


Saved artifact at '/var/folders/vs/xs37hl812y7_g9k12gvs5tww0000gn/T/tmp4q0m5rwp'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 2), dtype=tf.float32, name='keras_tensor')
Output Type:
  TensorSpec(shape=(None, 1), dtype=tf.float32, name=None)
Captures:
  5617276240: TensorSpec(shape=(), dtype=tf.resource, name=None)
  5617278160: TensorSpec(shape=(), dtype=tf.resource, name=None)
  5617277200: TensorSpec(shape=(), dtype=tf.resource, name=None)
  5617276432: TensorSpec(shape=(), dtype=tf.resource, name=None)
  5617278736: TensorSpec(shape=(), dtype=tf.resource, name=None)
  5617276816: TensorSpec(shape=(), dtype=tf.resource, name=None)
Done! Generated circadian_model.h


W0000 00:00:1739819118.980441 4622564 tf_tfl_flatbuffer_helpers.cc:365] Ignored output_format.
W0000 00:00:1739819118.980461 4622564 tf_tfl_flatbuffer_helpers.cc:368] Ignored drop_control_dependency.
2025-02-17 14:05:18.980925: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /var/folders/vs/xs37hl812y7_g9k12gvs5tww0000gn/T/tmp4q0m5rwp
2025-02-17 14:05:18.981377: I tensorflow/cc/saved_model/reader.cc:52] Reading meta graph with tags { serve }
2025-02-17 14:05:18.981389: I tensorflow/cc/saved_model/reader.cc:147] Reading SavedModel debug info (if present) from: /var/folders/vs/xs37hl812y7_g9k12gvs5tww0000gn/T/tmp4q0m5rwp
I0000 00:00:1739819118.984013 4622564 mlir_graph_optimization_pass.cc:401] MLIR V1 optimization pass is not enabled
2025-02-17 14:05:18.984462: I tensorflow/cc/saved_model/loader.cc:236] Restoring SavedModel bundle.
2025-02-17 14:05:19.002174: I tensorflow/cc/saved_model/loader.cc:220] Running initialization op on SavedModel bundle at path: /var/folder

In [2]:
import tensorflow as tf
import numpy as np
from datetime import datetime, time

def create_circadian_dataset():
    """Create training data for circadian rhythm"""
    # Generate time points (24 hours * 60 minutes = 1440 points)
    hours = np.repeat(np.arange(24), 60)
    minutes = np.tile(np.arange(60), 24)
    
    # Normalize inputs to 0-1 range
    X = np.column_stack([
        hours / 23.0,       # Normalize hours
        minutes / 59.0      # Normalize minutes
    ])
    
    # Generate target intensities
    y = []
    for h, m in zip(hours, minutes):
        time_val = time(hour=h, minute=m)
        
        # Early morning (5:00-9:00): Gradual increase
        if time(5) <= time_val < time(9):
            progress = (h - 5 + m/60) / 4
            intensity = 0.3 + (0.7 * progress)
        
        # Mid-morning to afternoon (9:00-15:00): High intensity
        elif time(9) <= time_val < time(15):
            intensity = 1.0
        
        # Afternoon to evening (15:00-21:00): Gradual decrease
        elif time(15) <= time_val < time(21):
            progress = (h - 15 + m/60) / 6
            intensity = 1.0 - (0.8 * progress)
        
        # Night (21:00-5:00): Low intensity
        else:
            intensity = 0.2
        
        y.append(intensity)
    
    return X, np.array(y)

def create_model():
    """Create and compile the model"""
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(2,), dtype=tf.float32),
        tf.keras.layers.Dense(8, activation='relu'),
        tf.keras.layers.Dense(4, activation='relu'),
        tf.keras.layers.Dense(1, activation='sigmoid')
    ])
    
    model.compile(
        optimizer='adam',
        loss='mse',
        metrics=['mae']
    )
    
    return model

def convert_to_tflite(model):
    """Convert model to TFLite format without quantization"""
    converter = tf.lite.TFLiteConverter.from_keras_model(model)
    tflite_model = converter.convert()
    return tflite_model

def generate_arduino_header(tflite_model, filename="circadian_model.h"):
    """Generate Arduino header file with model data"""
    header = """
// This is a TensorFlow Lite model file for a circadian lighting controller
#ifndef CIRCADIAN_MODEL_H_
#define CIRCADIAN_MODEL_H_

const unsigned char model_data[] = {
"""
    
    # Convert model binary to C array
    hex_data = ["0x{:02x}".format(b) for b in tflite_model]
    chunk_size = 12
    chunks = [hex_data[i:i + chunk_size] for i in range(0, len(hex_data), chunk_size)]
    
    for chunk in chunks:
        line = "  " + ", ".join(chunk) + ",\n"
        header += line
    
    header += """};

const unsigned int model_data_len = """ + str(len(tflite_model)) + """;

#endif  // CIRCADIAN_MODEL_H_
"""
    
    with open(filename, "w") as f:
        f.write(header)

def main():
    # Create dataset
    print("Creating dataset...")
    X, y = create_circadian_dataset()
    
    # Create and train model
    print("Training model...")
    model = create_model()
    model.fit(X, y, epochs=100, batch_size=32, verbose=1)
    
    # Convert to TFLite (float32)
    print("Converting to TFLite...")
    tflite_model = convert_to_tflite(model)
    
    # Generate Arduino header
    print("Generating Arduino header...")
    generate_arduino_header(tflite_model, "circadian_model.h")
    
    print("Done! Model size:", len(tflite_model), "bytes")

if __name__ == "__main__":
    main()

Creating dataset...
Training model...
Epoch 1/100
45/45 ━━━━━━━━━━━━━━━━━━━━ 0s 784us/step - loss: 0.1227 - mae: 0.3188 
Epoch 2/100
45/45 ━━━━━━━━━━━━━━━━━━━━ 0s 783us/step - loss: 0.1114 - mae: 0.3063
Epoch 3/100
45/45 ━━━━━━━━━━━━━━━━━━━━ 0s 648us/step - loss: 0.1088 - mae: 0.3054
Epoch 4/100
45/45 ━━━━━━━━━━━━━━━━━━━━ 0s 642us/step - loss: 0.1046 - mae: 0.2984
Epoch 5/100
45/45 ━━━━━━━━━━━━━━━━━━━━ 0s 668us/step - loss: 0.1072 - mae: 0.3060
Epoch 6/100
45/45 ━━━━━━━━━━━━━━━━━━━━ 0s 639us/step - loss: 0.1003 - mae: 0.2928
Epoch 7/100
45/45 ━━━━━━━━━━━━━━━━━━━━ 0s 655us/step - loss: 0.0972 - mae: 0.2880
Epoch 8/100
45/45 ━━━━━━━━━━━━━━━━━━━━ 0s 663us/step - loss: 0.0961 - mae: 0.2882
Epoch 9/100
45/45 ━━━━━━━━━━━━━━━━━━━━ 0s 652us/step - loss: 0.0924 - mae: 0.2813
Epoch 10/100
45/45 ━━━━━━━━━━━━━━━━━━━━ 0s 643us/step - loss: 0.0902 - mae: 0.2792
Epoch 11/100
45/45 ━━━━━━━━━━━━━━━━━━━━ 0s 616us/step - loss: 0.0847 - mae: 0.2691
Epoch 12/100
45/45 ━━━━━━━━━━━━━━━━━━━━ 0s 641us/step - l

INFO:tensorflow:Assets written to: /var/folders/vs/xs37hl812y7_g9k12gvs5tww0000gn/T/tmpvm4nd3b3/assets


Saved artifact at '/var/folders/vs/xs37hl812y7_g9k12gvs5tww0000gn/T/tmpvm4nd3b3'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 2), dtype=tf.float32, name='keras_tensor_4')
Output Type:
  TensorSpec(shape=(None, 1), dtype=tf.float32, name=None)
Captures:
  5617283344: TensorSpec(shape=(), dtype=tf.resource, name=None)
  5617286992: TensorSpec(shape=(), dtype=tf.resource, name=None)
  5617284880: TensorSpec(shape=(), dtype=tf.resource, name=None)
  5617281424: TensorSpec(shape=(), dtype=tf.resource, name=None)
  5623933712: TensorSpec(shape=(), dtype=tf.resource, name=None)
  5623923152: TensorSpec(shape=(), dtype=tf.resource, name=None)
Generating Arduino header...
Done! Model size: 2328 bytes


W0000 00:00:1739823883.500890 4622564 tf_tfl_flatbuffer_helpers.cc:365] Ignored output_format.
W0000 00:00:1739823883.500901 4622564 tf_tfl_flatbuffer_helpers.cc:368] Ignored drop_control_dependency.
2025-02-17 15:24:43.501020: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /var/folders/vs/xs37hl812y7_g9k12gvs5tww0000gn/T/tmpvm4nd3b3
2025-02-17 15:24:43.501345: I tensorflow/cc/saved_model/reader.cc:52] Reading meta graph with tags { serve }
2025-02-17 15:24:43.501351: I tensorflow/cc/saved_model/reader.cc:147] Reading SavedModel debug info (if present) from: /var/folders/vs/xs37hl812y7_g9k12gvs5tww0000gn/T/tmpvm4nd3b3
2025-02-17 15:24:43.504447: I tensorflow/cc/saved_model/loader.cc:236] Restoring SavedModel bundle.
2025-02-17 15:24:43.522182: I tensorflow/cc/saved_model/loader.cc:220] Running initialization op on SavedModel bundle at path: /var/folders/vs/xs37hl812y7_g9k12gvs5tww0000gn/T/tmpvm4nd3b3
2025-02-17 15:24:43.528091: I tensorflow/cc/saved_model/loader.cc: